[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/39_ppo_loss.ipynb)

# 🔴 Hard: PPO Clipped Loss

Реализуйте **PPO (Proximal Policy Optimization) clipped surrogate loss**. PPO обновляет policy по данным, собранным старой policy, и ограничивает слишком выгодные на текущем batch изменения probability ratio.

> Это только policy surrogate. Полный PPO обычно также включает value loss, entropy bonus, GAE/returns, несколько эпох minibatch-обновлений и диагностику KL. Их добавлять в эту функцию не нужно.

Given:
- `new_logps`: current policy log-probs $(B,)$
- `old_logps`: old policy log-probs $(B,)$
- `advantages`: advantage estimates $(B,)$

`advantages > 0` означает: действие оказалось лучше baseline, его вероятность желательно повысить. `advantages < 0` — вероятность желательно снизить. Так как данные получены old policy, новая policy корректирует вклад через importance ratio.

Define the ratio

$$ r_i = \exp(\text{new\_logps}_i - \text{old\_logps}_i). $$

Then compute
- $L^{\text{unclipped}}_i = r_i A_i$
- $L^{\text{clipped}}_i = \operatorname{clip}(r_i, 1-\epsilon, 1+\epsilon) A_i$

The loss is the negative batch mean of the elementwise minimum:

$$
\mathcal{L}_\text{PPO} = -\mathbb{E}_i\big[\min(L^{\text{unclipped}}_i, L^{\text{clipped}}_i)\big].
$$

## Как читать clipping

При положительном A рост ratio выше `1+eps` перестаёт улучшать surrogate: минимум выбирает clipped ветвь. При отрицательном A знаки меняют порядок, и защита срабатывает при чрезмерном уменьшении ratio ниже `1-eps`. Поэтому недостаточно рассуждать о clipping только для положительных advantages.

Важно: PPO не просто жёстко clamp-ит ratio во всех случаях. Elementwise `min(unclipped, clipped)` ограничивает **выгодное** удаление от старой policy, но не скрывает изменение, которое ухудшает objective.

Пример: `clip_ratio=0.2`, `A=2`, `ratio=1.5`. Unclipped surrogate равен 3, clipped — `1.2*2=2.4`, используется 2.4. При `ratio=0.7` и том же положительном A минимум выберет 1.4, а не поднимет его до `0.8*2`: плохое изменение не маскируется.

Implementation notes: detach `old_logps` and `advantages` so gradients only flow through `new_logps`.

### Signature
```python
from torch import Tensor

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    """PPO clipped surrogate loss over a batch."""
```

## План реализации

1. Detach old log-probabilities и advantages.
2. Вычислите ratio устойчиво как `exp(new_logps - old_logps)`.
3. Постройте unclipped и clipped surrogate поэлементно.
4. Возьмите elementwise minimum, batch mean и смените знак для минимизируемого loss.

## Быстрые проверки

- При `new_logps == old_logps` ratio равен 1, а loss равен `-mean(advantages)`.
- Проверьте отдельно положительный A с ratio выше верхней границы.
- Проверьте отрицательный A с ratio ниже нижней границы.
- Градиенты появляются только у `new_logps`.

## Частые ошибки

- Использовать `max` вместо `min` до смены знака.
- Clamp-ить log-probability difference вместо probability ratio.
- Забыть отрицательный знак у loss.
- Не detach old policy/advantages и обучать неподвижные цели.

## Материалы для глубокого изучения

- [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347) — исходная статья.
- [TorchRL PPO tutorial](https://docs.pytorch.org/rl/main/tutorials/coding_ppo.html) — полный training loop с GAE и value/entropy terms.
- [TorchRL ClipPPOLoss](https://docs.pytorch.org/rl/stable/reference/generated/torchrl.objectives.ClipPPOLoss.html) — промышленный контракт и метрики clipping/KL.

## Вопросы для самопроверки

1. Почему для отрицательного advantage clipping действует с другой стороны ratio?
2. Почему берётся minimum двух surrogate?
3. Какие компоненты полного PPO отсутствуют в этом упражнении?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    pass  # -mean(min(r * adv, clamp(r, 1-clip, 1+clip) * adv)) with gradients only through new_logps


In [ ]:
# 🧪 Debug
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('ppo_loss')
